## Analysis process - Dianela

In [1]:
from pathlib import Path
import statsmodels.formula.api as smf

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
raw_data   = Path("G:/Mi unidad/Master_LSE/Git-hub/Economic_Observatory/Data/raw_data")
clean_data = Path("G:/Mi unidad/Master_LSE/Git-hub/Economic_Observatory/Data/cleaning_data")

In [33]:
analysis_data = pd.read_csv(clean_data / "local_indicators_with_LAD_all_sample_sector.csv")
analysis_data.head(50)

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,OBS_Value_Business,OBS_Value_Employment,growth_firms,growth_emp,Area Code,County_or_Unitary_Authority,...,Data accuracy [Happiness],Local Authority District [Anxiety],Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety],Observation Status [Anxiety],Data accuracy [Anxiety],Local Authority District [Net additions],"Net additions per 1,000 stock [Net additions]",Homicide Offences (per million population) [Homicide],% living in an area that has a devolution deal with a directly elected mayor [Devolution],_merge
0,2022,E06000001,Hartlepool,Advanced manufacturing,20.0,1485.0,-0.213574,0.295338,E06000001,Hartlepool,...,coefficient of variation is less than or equal...,NaN,3.48,NaN,coefficient of variation is greater than 5% bu...,NaN,11.0,NaN,NaN,both
1,2022,E06000001,Hartlepool,Creative Industries,100.0,785.0,-0.094410,0.032323,E06000001,Hartlepool,...,coefficient of variation is less than or equal...,NaN,3.48,NaN,coefficient of variation is greater than 5% bu...,NaN,11.0,NaN,NaN,both
2,2022,E06000001,Hartlepool,Defence sector,0.0,0.0,0.000000,0.000000,E06000001,Hartlepool,...,coefficient of variation is less than or equal...,NaN,3.48,NaN,coefficient of variation is greater than 5% bu...,NaN,11.0,NaN,NaN,both
3,2022,E06000001,Hartlepool,Digital and Technology,285.0,1395.0,-0.146126,-0.055725,E06000001,Hartlepool,...,coefficient of variation is less than or equal...,NaN,3.48,NaN,coefficient of variation is greater than 5% bu...,NaN,11.0,NaN,NaN,both
4,2022,E06000001,Hartlepool,Financial Services,55.0,265.0,0.000000,0.255620,E06000001,Hartlepool,...,coefficient of variation is less than or equal...,NaN,3.48,NaN,coefficient of variation is greater than 5% bu...,NaN,11.0,NaN,NaN,both
5,2022,E06000001,Hartlepool,Life Sciences,0.0,40.0,0.000000,0.000000,E06000001,Hartlepool,...,coefficient of variation is less than or equal...,NaN,3.48,NaN,coefficient of variation is greater than 5% bu...,NaN,11.0,NaN,NaN,both
6,2022,E06000001,Hartlepool,Professional and Business Services,640.0,2580.0,-0.103643,-0.264062,E06000001,Hartlepool,...,coefficient of variation is less than or equal...,NaN,3.48,NaN,coefficient of variation is greater than 5% bu...,NaN,11.0,NaN,NaN,both
7,2022,E06000002,Middlesbrough,Advanced manufacturing,35.0,715.0,0.325423,0.555087,E06000002,Middlesbrough,...,coefficient of variation is less than or equal...,NaN,3.07,NaN,coefficient of variation is greater than 5% bu...,NaN,9.0,NaN,NaN,both
8,2022,E06000002,Middlesbrough,Creative Industries,190.0,1890.0,-0.051031,0.160251,E06000002,Middlesbrough,...,coefficient of variation is less than or equal...,NaN,3.07,NaN,coefficient of variation is greater than 5% bu...,NaN,9.0,NaN,NaN,both
9,2022,E06000002,Middlesbrough,Defence sector,0.0,0.0,0.000000,0.000000,E06000002,Middlesbrough,...,coefficient of variation is less than or equal...,NaN,3.07,NaN,coefficient of variation is greater than 5% bu...,NaN,9.0,NaN,NaN,both


In [34]:
cols_to_drop = [col for col in analysis_data.columns if 
                col.startswith('Local Authority District') or
                col.startswith('Notes') or
                col.startswith('Data accuracy') or
                col.startswith('Observation Status') or
                col.startswith('Country') or
                col.startswith('Nation') or
                col.startswith('Total value of UK') or               
                col.startswith('Total UK') or 
                col.startswith('% living in') or 
                col.startswith('Homicide Offences')]
# Drop merged variable and pattern-matched columns
analysis_data = analysis_data.drop(columns=['_merge', 'Area Code'] + cols_to_drop)

print(f"Columns dropped: {len(cols_to_drop) + 2}")
print(f"Remaining columns: {analysis_data.shape[1]}")

Columns dropped: 60
Remaining columns: 50


In [35]:
# Columns to exclude from indicators
cols_to_exclude = ['YEAR', 'GEOGRAPHY_CODE', 'GEOGRAPHY_NAME', 
                   'IS8_SECTOR','OBS_Value_Business', 'OBS_Value_Employment',
                   'growth_firms', 'growth_emp']

x = analysis_data.drop(columns=cols_to_exclude)
y = analysis_data[cols_to_exclude]

In [42]:
x

,County_or_Unitary_Authority,Gross Value Added (GVA) per hour worked (£) [GVA per hour],Gross median weekly pay (£) [Weekly pay],"Employment rate, aged 16 to 64 years (%) [Employment rate]","Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]","Gross disposable household income, per head (£) [GDHI per head]",Count of births of new enterprises (control rounded to base 5) [New enterprises],Count of deaths of enterprises (control rounded to base 5) [Deaths of enterprises],Count of active enterprises (control rounded to base 5) [Active enterprises],Count of high growth enterprises (control rounded to base 5) [High growth enterprises],...,Proportion of children living with obesity at reception age (%) [Reception obesity],Proportion of children living with obesity at Year 6 age (%) [Year 6 obesity],"Proportion of adults living with obesity, aged 18 years and over (%) [Adult obesity]",Proportion of cancers diagnosed at stages 1 and 2 (%) [Cancer diagnosis],"Age-standardised mortality rate for those aged under 75 (per 100,000 population) [Under 75 mortality rate]",Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction],Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile],Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness],Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety],"Net additions per 1,000 stock [Net additions]"
0,Hartlepool,29.84,521.2,69.7,4.6,16934.0,280.0,315.0,2430.0,10.0,...,12.68293,27.46781,35.35587,49.21053,37.85297,7.29,7.75,7.57,3.48,11.0
1,Hartlepool,29.84,521.2,69.7,4.6,16934.0,280.0,315.0,2430.0,10.0,...,12.68293,27.46781,35.35587,49.21053,37.85297,7.29,7.75,7.57,3.48,11.0
2,Hartlepool,29.84,521.2,69.7,4.6,16934.0,280.0,315.0,2430.0,10.0,...,12.68293,27.46781,35.35587,49.21053,37.85297,7.29,7.75,7.57,3.48,11.0
3,Hartlepool,29.84,521.2,69.7,4.6,16934.0,280.0,315.0,2430.0,10.0,...,12.68293,27.46781,35.35587,49.21053,37.85297,7.29,7.75,7.57,3.48,11.0
4,Hartlepool,29.84,521.2,69.7,4.6,16934.0,280.0,315.0,2430.0,10.0,...,12.68293,27.46781,35.35587,49.21053,37.85297,7.29,7.75,7.57,3.48,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2480,NaN,NaN,540,74.1,3.8,18038.0,12090.0,11660.0,104520.0,400.0,...,NaN,NaN,NaN,NaN,NaN,7.4,7.69,7.35,3.3,NaN
2481,NaN,NaN,540,74.1,3.8,18038.0,12090.0,11660.0,104520.0,400.0,...,NaN,NaN,NaN,NaN,NaN,7.4,7.69,7.35,3.3,NaN
2482,NaN,NaN,540,74.1,3.8,18038.0,12090.0,11660.0,104520.0,400.0,...,NaN,NaN,NaN,NaN,NaN,7.4,7.69,7.35,3.3,NaN
2483,NaN,NaN,540,74.1,3.8,18038.0,12090.0,11660.0,104520.0,400.0,...,NaN,NaN,NaN,NaN,NaN,7.4,7.69,7.35,3.3,NaN


In [44]:
#With the previous code, I realised that some of the indicators were not numeric, so I will convert them to numeric:
cols_to_convert = [
    'Gross median weekly pay (£) [Weekly pay]',
    'Employment rate, aged 16 to 64 years (%) [Employment rate]',
    'Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]',
    'Proportion of the population aged 16 to 64 with NVQ3+ qualification (%) [Level 3+ qualifications]',
    'Percentage of adults that currently smoke cigarettes (%) [Smokers]',
    'Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction]',
    'Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile]',
    'Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness]',
    'Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]'
]
for col in cols_to_convert:
    x[col] = pd.to_numeric(x[col], errors='coerce')

print(x.dtypes)

County_or_Unitary_Authority                                                                                                                                                       object
Gross Value Added (GVA) per hour worked (£) [GVA per hour]                                                                                                                       float64
Gross median weekly pay (£) [Weekly pay]                                                                                                                                         float64
Employment rate, aged 16 to 64 years (%) [Employment rate]                                                                                                                       float64
Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]                                                                                                       float64
Gross disposable household income, per head (£) [GDHI per head]            

In [ ]:
x
x_controls = x.groupby("County_or_Unitary_Authority").mean(numeric_only=True).reset_index()


In [45]:
x_controls.head()

,County_or_Unitary_Authority,Gross Value Added (GVA) per hour worked (£) [GVA per hour],"Gross disposable household income, per head (£) [GDHI per head]",Count of births of new enterprises (control rounded to base 5) [New enterprises],Count of deaths of enterprises (control rounded to base 5) [Deaths of enterprises],Count of active enterprises (control rounded to base 5) [Active enterprises],Count of high growth enterprises (control rounded to base 5) [High growth enterprises],"Average travel time in minutes to reach nearest large employment centre (500 to 4999 jobs available), by public transport or walking (minutes) [Public transport to employer]","Average travel time in minutes to reach nearest large employment centre (500 to 4999 jobs available), by car (minutes) [Drive to employer]","Average travel time in minutes to reach nearest large employment centre (500 to 4999 jobs available), by cycle (minutes) [Cycle to employer]",...,Percentage of 5-year olds at 'expected level' in literacy early learning goals (%) [Early years literacy],Percentage of 5-year olds at 'expected level' in maths early learning goals (%) [Early years maths],Count of 19+ Further Education and Skills Learner Achievements (qualifications) [FE and skills achievements],"Apprenticeships started by adults aged 16+ based on home address (per 100,000 population) [Apprenticeship starts]","19+ further education and skills participation (per 100,000 population) [FE and skills participation]",Proportion of children living with obesity at reception age (%) [Reception obesity],Proportion of children living with obesity at Year 6 age (%) [Year 6 obesity],"Proportion of adults living with obesity, aged 18 years and over (%) [Adult obesity]","Age-standardised mortality rate for those aged under 75 (per 100,000 population) [Under 75 mortality rate]","Net additions per 1,000 stock [Net additions]"
0,Bath and North East Somerset,28.45,23101.0,835.0,945.0,8980.0,50.0,12.495296,7.943799,10.514685,...,74.3,81.5,1730.0,766.0,3013.0,7.44048,17.30205,21.68340,20.12109,6.0
1,Bedford,32.74,21328.0,965.0,1025.0,8075.0,30.0,11.587914,7.952530,10.505185,...,69.4,76.5,3000.0,876.0,4930.0,8.33333,21.47806,24.02529,29.30756,18.0
2,Blackburn with Darwen,28.48,15025.0,815.0,855.0,6270.0,35.0,9.404357,7.019322,8.619506,...,65.9,72.2,3550.0,1067.0,8923.0,10.35354,24.72406,22.97324,48.34618,9.0
3,Blackpool,28.32,16717.0,795.0,565.0,4670.0,20.0,8.375514,6.636762,7.819612,...,66.2,72.5,3240.0,1222.0,7782.0,11.98630,26.92308,35.62741,52.62189,3.0
4,"Bournemouth, Christchurch and Poole",34.79,21751.0,1935.0,1935.0,17220.0,75.0,NaN,NaN,NaN,...,71.8,79.9,5970.0,929.0,4906.0,6.90162,18.96104,27.92590,27.00597,4.0


In [24]:
threshold = 0.1  # drop columns with more than 10% missing
missing_pct = analysis_data.isnull().mean()  # gives proportion (0 to 1)
cols_to_keep = missing_pct[missing_pct < threshold].index.tolist()
cols_to_drop = missing_pct[missing_pct >= threshold].index.tolist()

print(f"Columns kept:    {len(cols_to_keep)}")
print(f"Columns dropped: {len(cols_to_drop)}")
print(f"\nDropped columns:\n{cols_to_drop}")

# Apply
X_clean = analysis_data[cols_to_keep]

X_clean.head()

Columns kept:    32
Columns dropped: 79

Dropped columns:
['County_or_Unitary_Authority', 'Local Authority District [GVA per hour]', 'Country', 'Nation', 'Local Authority District [Weekly pay]', 'Local Authority District [Employment rate]', 'Local Authority District [Unemployment rate]', 'Local Authority District [GDHI per head]', 'Local Authority District [New enterprises]', 'Local Authority District [Deaths of enterprises]', 'Local Authority District [Active enterprises]', 'Local Authority District [High growth enterprises]', 'Total value of UK exports (£ million) [UK exports]', 'Total UK public-funded gross regional capital and non-capital expenditure on research and development (£ million) [Goverment R&D]', 'Local Authority District [Public transport to employer]', 'Average travel time in minutes to reach nearest large employment centre (500 to 4999 jobs available), by public transport or walking (minutes) [Public transport to employer]', 'Local Authority District [Drive to employe

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,FRONTIER_SECTOR,OBS_Value_Business,OBS_Value_Employment,growth_firms,growth_emp,Area Code,...,Percentage of adults that currently smoke cigarettes (%) [Smokers],Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction],Data accuracy [Life satisfaction],Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile],Data accuracy [Worthwhile],Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness],Data accuracy [Happiness],Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety],Data accuracy [Anxiety],_merge
0,2022,E06000001,Hartlepool,Professional and Business Services,"Accounting, audit and tax consultancy",25.0,175.0,0.000000,0.839751,E06000001,...,14.3,7.29,coefficient of variation is less than or equal...,7.75,coefficient of variation is less than or equal...,7.57,coefficient of variation is less than or equal...,3.48,coefficient of variation is greater than 5% bu...,both
1,2022,E06000001,Hartlepool,Creative Industries,Advertising and marketing,10.0,15.0,0.000000,-1.558145,E06000001,...,14.3,7.29,coefficient of variation is less than or equal...,7.75,coefficient of variation is less than or equal...,7.57,coefficient of variation is less than or equal...,3.48,coefficient of variation is greater than 5% bu...,both
2,2022,E06000001,Hartlepool,Advanced manufacturing,Aerospace manufacturing,0.0,0.0,0.000000,0.000000,E06000001,...,14.3,7.29,coefficient of variation is less than or equal...,7.75,coefficient of variation is less than or equal...,7.57,coefficient of variation is less than or equal...,3.48,coefficient of variation is greater than 5% bu...,both
3,2022,E06000001,Hartlepool,Advanced manufacturing,Agritech,0.0,0.0,0.000000,0.000000,E06000001,...,14.3,7.29,coefficient of variation is less than or equal...,7.75,coefficient of variation is less than or equal...,7.57,coefficient of variation is less than or equal...,3.48,coefficient of variation is greater than 5% bu...,both
4,2022,E06000001,Hartlepool,Financial Services,Asset management and wholesale services,10.0,45.0,-0.374693,0.245122,E06000001,...,14.3,7.29,coefficient of variation is less than or equal...,7.75,coefficient of variation is less than or equal...,7.57,coefficient of variation is less than or equal...,3.48,coefficient of variation is greater than 5% bu...,both


In [26]:
#Variables with growth rates:
X_clean= X_clean.rename(columns={
    'growth_firms': 'Business_growth_rate',
    'growth_emp': 'Employment_growth_rate'
})

In [27]:
X_clean.head()

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,FRONTIER_SECTOR,OBS_Value_Business,OBS_Value_Employment,Business_growth_rate,Employment_growth_rate,Gross Value Added (GVA) per hour worked (£) [GVA per hour],...,Count of active enterprises (control rounded to base 5) [Active enterprises],Count of high growth enterprises (control rounded to base 5) [High growth enterprises],Percentage of premises with gigabit-capable broadband (%) [Broadband availability],Percentage of 4G coverage by at least one mobile network operator (%) [4G area coverage],Proportion of the population aged 16 to 64 with NVQ3+ qualification (%) [Level 3+ qualifications],Percentage of adults that currently smoke cigarettes (%) [Smokers],Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction],Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile],Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness],Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]
0,2022,E06000001,Hartlepool,Professional and Business Services,"Accounting, audit and tax consultancy",25.0,175.0,0.000000,0.839751,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
1,2022,E06000001,Hartlepool,Creative Industries,Advertising and marketing,10.0,15.0,0.000000,-1.558145,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
2,2022,E06000001,Hartlepool,Advanced manufacturing,Aerospace manufacturing,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
3,2022,E06000001,Hartlepool,Advanced manufacturing,Agritech,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
4,2022,E06000001,Hartlepool,Financial Services,Asset management and wholesale services,10.0,45.0,-0.374693,0.245122,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48


In [28]:
#With the previous code, I realised that some of the indicators were not numeric, so I will convert them to numeric:
cols_to_convert = [
    'Gross median weekly pay (£) [Weekly pay]',
    'Employment rate, aged 16 to 64 years (%) [Employment rate]',
    'Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]',
    'Proportion of the population aged 16 to 64 with NVQ3+ qualification (%) [Level 3+ qualifications]',
    'Percentage of adults that currently smoke cigarettes (%) [Smokers]',
    'Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction]',
    'Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile]',
    'Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness]',
    'Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]'
]
for col in cols_to_convert:
    X_clean[col] = pd.to_numeric(X_clean[col], errors='coerce')

print(X_clean.dtypes)

YEAR                                                                                                       int64
GEOGRAPHY_CODE                                                                                            object
GEOGRAPHY_NAME                                                                                            object
IS8_SECTOR                                                                                                object
FRONTIER_SECTOR                                                                                           object
OBS_Value_Business                                                                                       float64
OBS_Value_Employment                                                                                     float64
Business_growth_rate                                                                                     float64
Employment_growth_rate                                                                          

In [29]:
##Number of missing values after conversion
print(X_clean.isnull().sum())

YEAR                                                                                                       0
GEOGRAPHY_CODE                                                                                             0
GEOGRAPHY_NAME                                                                                             0
IS8_SECTOR                                                                                                 0
FRONTIER_SECTOR                                                                                            0
OBS_Value_Business                                                                                         0
OBS_Value_Employment                                                                                       0
Business_growth_rate                                                                                       0
Employment_growth_rate                                                                                     0
Gross Value Added (

In [30]:
total = len(X_clean)
missing_pct = (X_clean.isnull().sum() / total * 100).round(1)
print(missing_pct[missing_pct > 0])

Gross Value Added (GVA) per hour worked (£) [GVA per hour]                                               2.5
Gross median weekly pay (£) [Weekly pay]                                                                 0.6
Employment rate, aged 16 to 64 years (%) [Employment rate]                                               1.7
Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]                               5.9
Gross disposable household income, per head (£) [GDHI per head]                                          1.1
Count of births of new enterprises (control rounded to base 5) [New enterprises]                         1.1
Count of deaths of enterprises (control rounded to base 5) [Deaths of enterprises]                       1.1
Count of active enterprises (control rounded to base 5) [Active enterprises]                             1.1
Count of high growth enterprises (control rounded to base 5) [High growth enterprises]                   1.1
Percentage of premi

In [31]:
# How many LADs needed the national median fallback?
lads_with_all_missing = X_clean.groupby('GEOGRAPHY_CODE')[col].apply(
    lambda x: x.isnull().all()
).sum()

print(f"LADs with no data at all for {col}: {lads_with_all_missing}")

LADs with no data at all for Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]: 16


One important thing — the wellbeing variables (Life satisfaction, Happiness, Worthwhile, Anxiety) and Unemployment rate tend to be missing in the same 16-21 LADs. This is likely because smaller or more rural LADs don't report these statistics.

For approximately 16-21 LADs (~5%) where no local data was available, missing values were imputed using the national median. These LADs are predominantly smaller or rural authorities with limited data reporting.

In [32]:
# Columns to exclude from indicators
cols_to_exclude = ['YEAR', 'GEOGRAPHY_CODE', 'GEOGRAPHY_NAME', 
                   'IS8_SECTOR', 'FRONTIER_SECTOR', 
                   'OBS_Value_Business', 'OBS_Value_Employment',
                   'Business_growth_rate', 'Employment_growth_rate']

# X — only local indicators
indicators = X_clean.drop(columns=cols_to_exclude)

In [33]:
##Imputations
fallback_summary = {}
for col in indicators.columns:
    lads_all_missing = X_clean.groupby('GEOGRAPHY_CODE')[col].apply(
        lambda x: x.isnull().all()
    ).sum()
    fallback_summary[col] = lads_all_missing
    
fallback_df = pd.DataFrame.from_dict(
    fallback_summary, orient='index', columns=['LADs using national median']
)

fallback_df = fallback_df[fallback_df['LADs using national median'] > 0].sort_values(
    'LADs using national median', ascending=False
)
print(fallback_df)
print(f"Total LADs in dataset: {X_clean['GEOGRAPHY_CODE'].nunique()}")

                                                    LADs using national median
Modelled unemployment rate, aged 16 years and o...                          21
Mean satisfaction with your life nowadays score...                          16
Mean feeling things done in life are worthwhile...                          16
Mean happiness yesterday scored 0 (not at all) ...                          16
Mean anxiety yesterday scored 0 (not at all) - ...                          16
Gross Value Added (GVA) per hour worked (£) [GV...                           9
Percentage of 4G coverage by at least one mobil...                           9
Percentage of premises with gigabit-capable bro...                           9
Percentage of adults that currently smoke cigar...                           7
Employment rate, aged 16 to 64 years (%) [Emplo...                           6
Proportion of the population aged 16 to 64 with...                           6
Count of births of new enterprises (control rou...  

In [ ]:
## I will drop of my analysis modelled unemployment rate, as I have the employment variable:




In [34]:
for col in indicators.columns:
    X_clean[col] = X_clean.groupby('GEOGRAPHY_CODE')[col].transform(
        lambda x: x.fillna(x.median())
    )
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())  # fallback for any remaining missing with the national median 

In [35]:
X_clean.head(50)

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,FRONTIER_SECTOR,OBS_Value_Business,OBS_Value_Employment,Business_growth_rate,Employment_growth_rate,Gross Value Added (GVA) per hour worked (£) [GVA per hour],...,Count of active enterprises (control rounded to base 5) [Active enterprises],Count of high growth enterprises (control rounded to base 5) [High growth enterprises],Percentage of premises with gigabit-capable broadband (%) [Broadband availability],Percentage of 4G coverage by at least one mobile network operator (%) [4G area coverage],Proportion of the population aged 16 to 64 with NVQ3+ qualification (%) [Level 3+ qualifications],Percentage of adults that currently smoke cigarettes (%) [Smokers],Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction],Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile],Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness],Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]
0,2022,E06000001,Hartlepool,Professional and Business Services,"Accounting, audit and tax consultancy",25.0,175.0,0.000000,0.839751,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
1,2022,E06000001,Hartlepool,Creative Industries,Advertising and marketing,10.0,15.0,0.000000,-1.558145,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
2,2022,E06000001,Hartlepool,Advanced manufacturing,Aerospace manufacturing,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
3,2022,E06000001,Hartlepool,Advanced manufacturing,Agritech,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
4,2022,E06000001,Hartlepool,Financial Services,Asset management and wholesale services,10.0,45.0,-0.374693,0.245122,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
5,2022,E06000001,Hartlepool,Advanced manufacturing,Automotive manufacturing,0.0,400.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
6,2022,E06000001,Hartlepool,Advanced manufacturing,Batteries,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
7,2022,E06000001,Hartlepool,Financial Services,Capital markets and retail investment,20.0,90.0,0.271934,0.056512,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
8,2022,E06000001,Hartlepool,Defence sector,Defence Sector,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
9,2022,E06000001,Hartlepool,Digital and Technology,Digital and Technology,285.0,1395.0,-0.146126,-0.055725,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48


In [ ]:
lad_vars = (
    X_clean[X_clean["FRONTIER_SECTOR"] == "Total"]
    .groupby("GEOGRAPHY_CODE")
    .agg(
        GEOGRAPHY_NAME  = ("GEOGRAPHY_NAME", "first"),
        Business_growth = ("Business_growth_rate", "first"),  # adjust col name
        Employ_growth   = ("Employ_growth_rate",   "first"),  # adjust col name
        total_bus       = ("OBS_Value_Business", "first"),
        total_emp       = ("OBS_Value_Employment",  "first"),
    )
    .reset_index()
)

# 1b. Calculate sector shares from non-Total rows
sectors = X_clean[X_clean["FRONTIER_SECTOR"] != "Total"].copy()
sectors = sectors.merge(
    lad_vars[["GEOGRAPHY_CODE", "total_bus", "total_emp"]],
    on="GEOGRAPHY_CODE"
)

# Enterprise share per sector
sectors["ent_share"] = sectors["OBS_Value_Business"] / sectors["total_bus"]
# Employment share per sector
sectors["emp_share"] = sectors["OBS_Value_Employment"]  / sectors["total_emp"]

# 1c. Pivot to wide — one column per Frontier sector
ent_wide = sectors.pivot_table(
    index="GEOGRAPHY_CODE", columns="FRONTIER_SECTOR", values="ent_share"
)
ent_wide.columns = [f"ent_share_{c}" for c in ent_wide.columns]

emp_wide = sectors.pivot_table(
    index="GEOGRAPHY_CODE", columns="FRONTIER_SECTOR", values="emp_share"
)
emp_wide.columns = [f"emp_share_{c}" for c in emp_wide.columns]

# 1d. Merge everything together
df_wide = (
    lad_vars
    .set_index("GEOGRAPHY_CODE")
    .join(ent_wide)
    .join(emp_wide)
    .reset_index()
)

print(f"Wide dataset shape: {df_wide.shape}")  # should be ~350 rows
print(df_wide.head())


In [36]:
ddddd

NameError: name 'ddddd' is not defined

In [ ]:
X = X_clean[indicators.columns].copy()
y_bus = X_clean['Business_growth_rate'].reset_index(drop=True)
y_emp = X_clean['Employment_growth_rate'].reset_index(drop=True)

print(f'X shape: {X.shape}, y_business growth shape: {y_bus.shape}, y_employment growth shape: {y_emp.shape}')

X shape: (7810, 17), y_business growth shape: (7810,), y_employment growth shape: (7810,)


In [ ]:
# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=indicators.columns, 
                            index=X_clean.index)

# Sector dummies
is8_dummies     = pd.get_dummies(X_clean['IS8_SECTOR'], 
                                  prefix='IS8', drop_first=True)
frontier_dummies = pd.get_dummies(X_clean['FRONTIER_SECTOR'], 
                                   prefix='FS', drop_first=True)

In [ ]:
cddddd

In [ ]:
##### LASSO — variable selection 
lasso_bus = LassoCV(cv=10, random_state=42, max_iter=10000).fit(X_scaled, y_bus)
lasso_emp = LassoCV(cv=10, random_state=42, max_iter=10000).fit(X_scaled, y_emp)

selected_bus = [col for col, coef in zip(indicators.columns, lasso_bus.coef_) if coef != 0]
selected_emp = [col for col, coef in zip(indicators.columns, lasso_emp.coef_) if coef != 0]

# LASSO coefficients summary
lasso_bus_df = pd.DataFrame({
    'Variable'   : indicators.columns,
    'Coefficient': lasso_bus.coef_
}).query('Coefficient != 0').sort_values('Coefficient', key=abs, ascending=False)

lasso_emp_df = pd.DataFrame({
    'Variable'   : indicators.columns,
    'Coefficient': lasso_emp.coef_
}).query('Coefficient != 0').sort_values('Coefficient', key=abs, ascending=False)


C:\Users\diane\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.903e+00, tolerance: 2.159e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\diane\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.078e+01, tolerance: 2.159e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\diane\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or c

In [ ]:
print(f"LASSO — Business growth selected {len(selected_bus)} variables:")
print(lasso_bus_df.to_string(index=False))

LASSO — Business growth selected 8 variables:
                                                                                             Variable  Coefficient
                                           Gross Value Added (GVA) per hour worked (£) [GVA per hour]    -0.009159
                                                             Gross median weekly pay (£) [Weekly pay]    -0.007358
    Proportion of the population aged 16 to 64 with NVQ3+ qualification (%) [Level 3+ qualifications]     0.006564
                             Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]    -0.004866
                   Percentage of premises with gigabit-capable broadband (%) [Broadband availability]    -0.004678
                                   Percentage of adults that currently smoke cigarettes (%) [Smokers]     0.004444
                                           Employment rate, aged 16 to 64 years (%) [Employment rate]     0.002566
Mean satisfaction with your life n

In [ ]:
print(f"LASSO — Employment growth selected {len(selected_emp)} variables:")
print(lasso_emp_df.to_string(index=False))

LASSO — Employment growth selected 0 variables:
Empty DataFrame
Columns: [Variable, Coefficient]
Index: []


In [ ]:
# ── Sector dummies ────────────────────────────────────────────────
is8_dummies      = pd.get_dummies(X_clean['IS8_SECTOR'],      drop_first=True).astype(float)
frontier_dummies = pd.get_dummies(X_clean['FRONTIER_SECTOR'], drop_first=True).astype(float)

In [ ]:
# ── Modelos CON estandarización ───────────────────────────────────
def run_ols_scaled(y, dummies, label):
    X_ols = pd.concat([
        X_scaled_df[indicators.columns].reset_index(drop=True),
        dummies.reset_index(drop=True)
    ], axis=1).astype(float)
    X_ols = sm.add_constant(X_ols)
    model = sm.OLS(y, X_ols).fit(cov_type='HC3')
    print(f"\n{'='*60}")
    print(f"MODEL (SCALED): {label}")
    print(model.summary())
    return model

# ── Modelos SIN estandarización ───────────────────────────────────
def run_ols_raw(y, dummies, label):
    X_ols = pd.concat([
        X_clean[indicators.columns].reset_index(drop=True),
        dummies.reset_index(drop=True)
    ], axis=1).astype(float)
    X_ols = sm.add_constant(X_ols)
    model = sm.OLS(y, X_ols).fit(cov_type='HC3')
    print(f"\n{'='*60}")
    print(f"MODEL (RAW): {label}")
    print(model.summary())
    return model

### IS8 FIXED EFFECTS

Business outcome

In [ ]:

print("IS8 with standarized local indicators")
m1_scaled = run_ols_scaled(y_bus, is8_dummies, "M1: Business Growth   + IS8 FE")

IS8 with standarized local indicators

MODEL (SCALED): M1: Business Growth   + IS8 FE
                             OLS Regression Results                             
Dep. Variable:     Business_growth_rate   R-squared:                       0.253
Model:                              OLS   Adj. R-squared:                  0.250
Method:                   Least Squares   F-statistic:                     6.821
Date:                  Sun, 15 Mar 2026   Prob (F-statistic):           1.38e-21
Time:                          15:45:56   Log-Likelihood:                -61410.
No. Observations:                  6035   AIC:                         1.229e+05
Df Residuals:                      6011   BIC:                         1.230e+05
Df Model:                            23                                         
Covariance Type:                    HC3                                         
                                                                                                        

In [ ]:
print("IS8 without standarized local indicators")
m1_raw    = run_ols_raw   (y_bus, is8_dummies, "M1: Business Growth   + IS8 FE")

IS8 without standarized local indicators



MODEL (RAW): M1: Business Growth   + IS8 FE
                             OLS Regression Results                             
Dep. Variable:     Business_growth_rate   R-squared:                       0.253
Model:                              OLS   Adj. R-squared:                  0.250
Method:                   Least Squares   F-statistic:                     6.821
Date:                  Sun, 15 Mar 2026   Prob (F-statistic):           1.38e-21
Time:                          15:45:56   Log-Likelihood:                -61410.
No. Observations:                  6035   AIC:                         1.229e+05
Df Residuals:                      6011   BIC:                         1.230e+05
Df Model:                            23                                         
Covariance Type:                    HC3                                         
                                                                                                            coef    std err          z      P>|z|

Employment outcome

In [ ]:
#With standarizaed local indicators
m2_scaled = run_ols_scaled(y_emp, is8_dummies, "M2: Employment Growth + IS8 FE")



MODEL (SCALED): M2: Employment Growth + IS8 FE
                              OLS Regression Results                              
Dep. Variable:     Employment_growth_rate   R-squared:                       0.295
Model:                                OLS   Adj. R-squared:                  0.292
Method:                     Least Squares   F-statistic:                     11.13
Date:                    Sun, 15 Mar 2026   Prob (F-statistic):           2.88e-40
Time:                            15:45:56   Log-Likelihood:                -73160.
No. Observations:                    6035   AIC:                         1.464e+05
Df Residuals:                        6011   BIC:                         1.465e+05
Df Model:                              23                                         
Covariance Type:                      HC3                                         
                                                                                                            coef    std er

In [ ]:
#Without standarised local indicators
m2_raw    = run_ols_raw   (y_emp, is8_dummies, "M2: Employment Growth + IS8 FE")


MODEL (RAW): M2: Employment Growth + IS8 FE
                              OLS Regression Results                              
Dep. Variable:     Employment_growth_rate   R-squared:                       0.295
Model:                                OLS   Adj. R-squared:                  0.292
Method:                     Least Squares   F-statistic:                     11.13
Date:                    Sun, 15 Mar 2026   Prob (F-statistic):           2.88e-40
Time:                            15:45:56   Log-Likelihood:                -73160.
No. Observations:                    6035   AIC:                         1.464e+05
Df Residuals:                        6011   BIC:                         1.465e+05
Df Model:                              23                                         
Covariance Type:                      HC3                                         
                                                                                                            coef    std err  

## FRONTIER SECTOR FIXED EFFECTS

Business outcome

In [ ]:
# With standarised local indicators
m3_scaled = run_ols_scaled(y_bus, frontier_dummies, "M3: Business Growth   + Frontier FE")



MODEL (SCALED): M3: Business Growth   + Frontier FE
                             OLS Regression Results                             
Dep. Variable:     Business_growth_rate   R-squared:                       0.255
Model:                              OLS   Adj. R-squared:                  0.251
Method:                   Least Squares   F-statistic:                     31.21
Date:                  Sun, 15 Mar 2026   Prob (F-statistic):          7.68e-179
Time:                          15:45:56   Log-Likelihood:                -61405.
No. Observations:                  6035   AIC:                         1.229e+05
Df Residuals:                      6001   BIC:                         1.231e+05
Df Model:                            33                                         
Covariance Type:                    HC3                                         
                                                                                                            coef    std err          z   

In [ ]:
# Without standarised local indicators
m3_raw    = run_ols_raw   (y_bus, frontier_dummies, "M3: Business Growth   + Frontier FE")


MODEL (RAW): M3: Business Growth   + Frontier FE
                             OLS Regression Results                             
Dep. Variable:     Business_growth_rate   R-squared:                       0.255
Model:                              OLS   Adj. R-squared:                  0.251
Method:                   Least Squares   F-statistic:                     31.21
Date:                  Sun, 15 Mar 2026   Prob (F-statistic):          7.70e-179
Time:                          15:45:56   Log-Likelihood:                -61405.
No. Observations:                  6035   AIC:                         1.229e+05
Df Residuals:                      6001   BIC:                         1.231e+05
Df Model:                            33                                         
Covariance Type:                    HC3                                         
                                                                                                            coef    std err          z      

Employment outcome

In [ ]:
m4_scaled = run_ols_scaled(y_emp, frontier_dummies, "M4: Employment Growth + Frontier FE")


MODEL (SCALED): M4: Employment Growth + Frontier FE
                              OLS Regression Results                              
Dep. Variable:     Employment_growth_rate   R-squared:                       0.295
Model:                                OLS   Adj. R-squared:                  0.291
Method:                     Least Squares   F-statistic:                     21.08
Date:                    Sun, 15 Mar 2026   Prob (F-statistic):          1.10e-117
Time:                            15:45:57   Log-Likelihood:                -73159.
No. Observations:                    6035   AIC:                         1.464e+05
Df Residuals:                        6001   BIC:                         1.466e+05
Df Model:                              33                                         
Covariance Type:                      HC3                                         
                                                                                                            coef    s

In [ ]:
m4_raw    = run_ols_raw   (y_emp, frontier_dummies, "M4: Employment Growth + Frontier FE")


MODEL (RAW): M4: Employment Growth + Frontier FE
                              OLS Regression Results                              
Dep. Variable:     Employment_growth_rate   R-squared:                       0.295
Model:                                OLS   Adj. R-squared:                  0.291
Method:                     Least Squares   F-statistic:                     21.08
Date:                    Sun, 15 Mar 2026   Prob (F-statistic):          1.11e-117
Time:                            15:45:57   Log-Likelihood:                -73159.
No. Observations:                    6035   AIC:                         1.464e+05
Df Residuals:                        6001   BIC:                         1.466e+05
Df Model:                              33                                         
Covariance Type:                      HC3                                         
                                                                                                            coef    std 